In [1]:
# Now to download moving rent data from 00 to 25 :)

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl

In [2]:
Current_Path = Path.cwd()

Path_Root = Current_Path.parent.parent

Data_Path = Path_Root/"data"/"forecasting"
File_Path = Data_Path/"MovingRent_0025.xlsx"

Data_Path.mkdir(parents=True, exist_ok=True)
Url = (
    "https://www.dffh.vic.gov.au/"
    "moving-annual-rent-suburb-september-quarter-2025-excel"
)

Response = requests.get(
    Url,
    timeout=60,
)

Response.raise_for_status()

with open(File_Path, "wb") as File:
    File.write(Response.content)


print(f"Downloaded successfully to: {File_Path}")

Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/forecasting/MovingRent_0025.xlsx


In [6]:
# Now to csv this shit!!!

Moving_Rent_Data = pd.read_excel(
    File_Path
)

Csv1_File_Path = Data_Path/"MovingRent_0025_Uncleaned.csv"

Moving_Rent_Data.to_csv(Csv1_File_Path,
                    index=False,
                    header=False
                    )

In [ ]:
# Time to read this data in and clean it up

Uncleaned_Data = pd.read_csv(Csv1_File_Path, header=[0,1])
Uncleaned_Data

One bedroom flat                     Unnamed: 1_level_0 Mar 2000         \
    Unnamed: 0_level_1                     Unnamed: 1_level_1    Count Median   
0      Inner Melbourne  Albert Park-Middle Park-West St Kilda      352    165   
1                  NaN                               Armadale      210    150   
2                  NaN                          Carlton North       87    150   
3                  NaN                      Carlton-Parkville      298    165   
4                  NaN                        CBD-St Kilda Rd      755    250   
..                 ...                                    ...      ...    ...   
154                NaN                            Wanagaratta       51     85   
155                NaN                               Warragul       13     80   
156                NaN                            Warrnambool      113     75   
157                NaN                                Wodonga       77     85   
158                NaN                            Group Total     1132     80   

    Jun 2000        Sep 2000        Dec 2000         ... Sep 2024         \
       Count Median    Count Median    Count Median  ...    Count Median   
0        347    165      378    170      369    175  ...      180    440   
1        212    150      213    155      213    160  ...      152    460   
2         78    155       74    150       65    150  ...       48    410   
3        297    170      312    175      346    180  ...     1129    480   
4        861    250      934    250      952    250  ...     6029    550   
..       ...    ...      ...    ...      ...    ...  ...      ...    ...   
154       46     85       44     85       47     85  ...       51    275   
155       11     75       12     90       10     90  ...       14    268   
156      104     75      108     75      105     80  ...       42    300   
157       72     85       77     85       83     85  ...       47    260   
158     1125     80     1148     80     1148     80  ...      863    260   

    Dec 2024        Mar 2025        Jun 2025        Sep 2025         
       Count Median    Count Median    Count Median    Count Median  
0        166    450      178    450      186    450      176    450  
1        169    450      178    450      181    460      188    450  
2         44    420       45    420       37    425       37    430  
3       1101    490     1106    500     1048    500     1023    500  
4       6008    550     6246    550     6277    559     6274    560  
..       ...    ...      ...    ...      ...    ...      ...    ...  
154       46    285       61    280       65    280       68    267  
155       11    250       12    251        -      -       10    250  
156       46    320       37    325       46    335       40    345  
157       51    270       49    270       50    280       54    290  
158      870    260      822    270      813    275      783    278  

[159 rows x 208 columns]

In [ ]:
# First let me get the coloumns that i want, the suburb coloumns + medians

# The second column contains the suburb names.
Suburb_Column = Uncleaned_Data.columns[1]

# Find all columns where the second header is "Median".
Median_Columns = [
    Column
    for Column in Uncleaned_Data.columns
    if str(Column[1]).strip() == "Median"
]

# Keep suburb + median columns.
Cleaned_Data = Uncleaned_Data[
    [Suburb_Column] + Median_Columns
].copy()



,Unnamed: 1_level_0,Mar 2000,Jun 2000,Sep 2000,Dec 2000,Mar 2001,Jun 2001,Sep 2001,Dec 2001,Mar 2002,...,Jun 2023,Sep 2023,Dec 2023,Mar 2024,Jun 2024,Sep 2024,Dec 2024,Mar 2025,Jun 2025,Sep 2025
,Unnamed: 1_level_1,Median,Median,Median,Median,Median,Median,Median,Median,Median,...,Median,Median,Median,Median,Median,Median,Median,Median,Median,Median
0,Albert Park-Middle Park-West St Kilda,165,165,170,175,180,185,190,190,195,...,370,395,400,425,428,440,450,450,450,450
1,Armadale,150,150,155,160,160,160,165,165,165,...,385,400,408,430,450,460,450,450,460,450
2,Carlton North,150,155,150,150,160,160,160,160,165,...,380,380,380,400,400,410,420,420,425,430
3,Carlton-Parkville,165,170,175,180,185,190,195,185,180,...,415,430,440,450,470,480,490,500,500,500
4,CBD-St Kilda Rd,250,250,250,250,255,260,260,260,265,...,479,500,520,550,550,550,550,550,559,560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,Wanagaratta,85,85,85,85,85,85,85,85,85,...,220,220,230,240,255,275,285,280,280,267
155,Warragul,80,75,90,90,80,80,80,80,-,...,-,-,260,260,295,268,250,251,-,250
156,Warrnambool,75,75,75,80,80,80,80,80,80,...,250,260,300,300,300,300,320,325,335,345


In [11]:
# Secondly lemme change the first coloum name to suburb

Cleaned_Data.columns = (
    ["Suburb"]
    + [
        Column[0]
        for Column in Median_Columns
    ]
)

In [12]:
######################################################

# Now to save my dataset yippie

######################################################

Csv2_File_Path = Data_Path/"MovingRent_0025_Cleaned.csv"

Cleaned_Data.to_csv(
    Csv2_File_Path,
    index=False,
)

print(Cleaned_Data.head())
print(
    f"Cleaned data saved to: {Csv2_File_Path}"
)

                                  Suburb Mar 2000 Jun 2000 Sep 2000 Dec 2000  \
0  Albert Park-Middle Park-West St Kilda      165      165      170      175   
1                               Armadale      150      150      155      160   
2                          Carlton North      150      155      150      150   
3                      Carlton-Parkville      165      170      175      180   
4                        CBD-St Kilda Rd      250      250      250      250   

  Mar 2001 Jun 2001 Sep 2001 Dec 2001 Mar 2002  ... Jun 2023 Sep 2023  \
0      180      185      190      190      195  ...      370      395   
1      160      160      165      165      165  ...      385      400   
2      160      160      160      160      165  ...      380      380   
3      185      190      195      185      180  ...      415      430   
4      255      260      260      260      265  ...      479      500   

  Dec 2023 Mar 2024 Jun 2024 Sep 2024 Dec 2024 Mar 2025 Jun 2025 Sep 2025  
0   